In [ ]:
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
OUTPUT_DIR = "../model_evaluation/summary"

PERTURBATION_SCENARIOS = [
    "gaussian_noise",
    "blur",
    "brightness_shift",
]
SEVERITY_LABELS = {
    ("clean", "default"): "Clean",
    ("gaussian_noise", "severity_level_1"): "$\\sigma_1=0.03$",
    ("gaussian_noise", "severity_level_2"): "$\\sigma_2=0.1$",
    ("gaussian_noise", "severity_level_3"): "$\\sigma_3=0.3$",
    ("blur", "severity_level_1"): "$k_1=3$",
    ("blur", "severity_level_2"): "$k_2=5$",
    ("blur", "severity_level_3"): "$k_3=7$",
    ("brightness_shift", "severity_level_1"): "$\\beta_1=-0.3$",
    ("brightness_shift", "severity_level_2"): "$\\beta_2=0.05$",
    ("brightness_shift", "severity_level_3"): "$\\beta_3=0.3$",
}
CHECKPOINT_TO_STEP_MAPPING = {
    "000000": 0,
    "004000": 4000,
    "008000": 8000,
    "012000": 12000,
    "016000": 16000,
    "020000": 20000,
    "024000": 24000,
    "last": 25650,
}

In [ ]:
success_rates = json.load(open(f"{OUTPUT_DIR}/success_rates.json", "r"))

## Success rate

In [ ]:
def get_success_rate_dataframe(scenario: str, severity_level: str) -> pd.DataFrame:

    df = pd.DataFrame(success_rates[scenario][severity_level])
    df.index = [CHECKPOINT_TO_STEP_MAPPING[ckpt] / 10000 for ckpt in df.index]

    max_success_rates = df.max(axis=1)
    mean_success_rates = df.mean(axis=1)
    min_success_rates = df.min(axis=1)

    df["max_success_rate"] = max_success_rates
    df["mean_success_rate"] = mean_success_rates
    df["min_success_rate"] = min_success_rates

    return df


def draw_success_rate(ax, df: pd.DataFrame, scenario: str, severity_level: str) -> None:

    sns.lineplot(
        data=df["mean_success_rate"] * 100,
        ax=ax,
        label=SEVERITY_LABELS[(scenario, severity_level)],
    )
    ax.fill_between(
        df.index,
        df["min_success_rate"] * 100,
        df["max_success_rate"] * 100,
        alpha=0.2,
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 3))

for i, scenario in enumerate(PERTURBATION_SCENARIOS):
    ax = axes[i]

    df_clean = get_success_rate_dataframe("clean", "default")
    draw_success_rate(ax, df_clean, "clean", "default")

    for severity_level in success_rates[scenario]:
        df_case = get_success_rate_dataframe(scenario, severity_level)
        draw_success_rate(ax, df_case, scenario, severity_level)

    ax.grid()
    ax.set_ylabel("")
    ax.set_xlabel("Training steps ($\\times 10^4$)")
    ax.set_title(scenario.replace("_", " ").title())

axes[0].set_ylabel("Successful Lifting Rate (%)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/success_rate.pdf")
plt.savefig(f"{OUTPUT_DIR}/success_rate.png")

## Success rate degradation

In [ ]:
def draw_success_rate_degradation(ax, df_clean: pd.DataFrame, df_case: pd.DataFrame, scenario: str, severity_level: str) -> None:

    sns.lineplot(
        data=(df_clean["mean_success_rate"] - df_case["mean_success_rate"]).clip(
            lower=0
        )
        * 100,
        ax=ax,
        label=SEVERITY_LABELS[(scenario, severity_level)],
    )
    ax.fill_between(
        df_case.index,
        (df_clean["mean_success_rate"] - df_case["max_success_rate"]).clip(lower=0)
        * 100,
        (df_clean["mean_success_rate"] - df_case["min_success_rate"]).clip(lower=0)
        * 100,
        alpha=0.2,
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 3))

for i, scenario in enumerate(PERTURBATION_SCENARIOS):
    ax = axes[i]

    df_clean = get_success_rate_dataframe("clean", "default")

    for severity_level in success_rates[scenario]:
        df_case = get_success_rate_dataframe(scenario, severity_level)
        draw_success_rate_degradation(ax, df_clean, df_case, scenario, severity_level)

    ax.grid()
    ax.set_ylabel("")
    ax.set_ylim(0, 35)
    ax.set_xlabel("Training steps ($\\times 10^4$)")
    ax.set_title(scenario.replace("_", " ").title())

axes[0].set_ylabel("Success Rate Degradation (%)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/success_rate_degradation.pdf")
plt.savefig(f"{OUTPUT_DIR}/success_rate_degradation.png")